**Capstone Project: Customer Order Insights & Delivery Tracker**

**Week 4 - Simple ETL Pipeline in Azure Databricks**

**Capstone Tasks:**

-  Load cleaned order data into Databricks
-  Create a pipeline to update latest delivery status
-  Save the results as Delta or CSV
-  Optional: run a SQL query to show top 5 delayed customers

In [0]:
# 1. Make sure the volume exists (does nothing if it is already there)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.customer_orders")

# 2. Write the CSV file into the volume
csv_text = """order_id,status,delivery_date,issue
3,Delivered,2026-10-09 11:20:00,Courier delay
8,Delayed,,Address not found
14,Delivered,2026-10-08 15:40:00,Stock shortage
19,Delivered,2026-10-09 17:10:00,
22,Delivered,2026-10-10 10:00:00,
"""

path = "/Volumes/workspace/default/customer_orders/delivery_updates.csv"
with open(path, "w") as f:
    f.write(csv_text)

# 3. Check it
display(spark.read.csv(path, header=True))

order_id,status,delivery_date,issue
3,Delivered,2026-10-09 11:20:00,Courier delay
8,Delayed,null,Address not found
14,Delivered,2026-10-08 15:40:00,Stock shortage
19,Delivered,2026-10-09 17:10:00,null
22,Delivered,2026-10-10 10:00:00,null


In [0]:
BASE = "/Volumes/workspace/default/customer_orders"

ORDERS_PATH    = f"{BASE}/cleaned_orders.csv"
CUSTOMERS_PATH = f"{BASE}/customers.csv"
UPDATES_PATH   = f"{BASE}/delivery_updates.csv"
REPORT_DATE = "2026-10-10" 

In [0]:
# 1. Extract - load the cleaned orders, customers and latest delivery updates
from pyspark.sql import functions as F
 
# Cleaned orders from Week 2 (read as text, then cast to the right types)
orders = (spark.read.csv(ORDERS_PATH, header=True)
    .withColumn("order_id",      F.col("order_id").cast("int"))
    .withColumn("customer_id",   F.col("customer_id").cast("int"))
    .withColumn("quantity",      F.col("quantity").cast("int"))
    .withColumn("amount",        F.col("amount").cast("double"))
    .withColumn("order_date",    F.col("order_date").cast("timestamp"))
    .withColumn("expected_date", F.col("expected_date").cast("date"))
    .withColumn("delivery_date", F.col("delivery_date").cast("timestamp"))
    .withColumn("delay_days",    F.col("delay_days").cast("int"))
    .withColumn("delayed",       F.col("delayed").cast("int")))
 
customers = spark.read.csv(CUSTOMERS_PATH, header=True, inferSchema=True)
display(customers)
 
# Latest delivery updates from the courier
updates = (spark.read.csv(UPDATES_PATH, header=True)
    .withColumn("order_id",          F.col("order_id").cast("int"))
    .withColumn("new_status",        F.trim(F.col("status")))
    .withColumn("new_delivery_date", F.col("delivery_date").cast("timestamp"))
    .withColumn("new_issue",         F.trim(F.col("issue")))
    .select("order_id", "new_status", "new_delivery_date", "new_issue"))
 
print("Orders:", orders.count(), "| Customers:", customers.count(), "| Updates:", updates.count())
display(updates)

customer_id,customer_name,city,region
1,Arun Kumar,Chennai,South
2,Priya Sharma,Delhi,North
3,Karthik Raj,Coimbatore,South
4,Sneha Patel,Mumbai,West
5,Rahul Das,Kolkata,East
6,Meena Iyer,Madurai,South
7,Amit Verma,Lucknow,North
8,Divya Nair,Pune,West


Orders: 22 | Customers: 8 | Updates: 5


order_id,new_status,new_delivery_date,new_issue
3,Delivered,2026-10-09T11:20:00.000Z,Courier delay
8,Delayed,null,Address not found
14,Delivered,2026-10-08T15:40:00.000Z,Stock shortage
19,Delivered,2026-10-09T17:10:00.000Z,null
22,Delivered,2026-10-10T10:00:00.000Z,null


In [0]:
# 2. Load the starting Delta tables (version 0)
(orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable("customer_orders_clean"))
(customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable("customer_customers"))
display(spark.table("customer_orders_clean").orderBy("order_id"))

order_id,customer_id,product_name,quantity,amount,order_date,expected_date,delivery_date,status,issue,delay_days,delayed
1,1,Wireless Mouse,1,799.0,2026-09-01T10:15:00.000Z,2026-09-06,2026-09-05T14:30:00.000Z,Delivered,null,0,0
2,1,Laptop Bag,1,1499.0,2026-09-10T09:40:00.000Z,2026-09-15,2026-09-18T16:05:00.000Z,Delayed,Courier Delay,3,1
3,1,USB Cable,3,450.0,2026-09-25T18:20:00.000Z,2026-09-30,null,Delayed,Courier Delay,10,1
4,2,Bluetooth Speaker,1,2499.0,2026-09-02T11:00:00.000Z,2026-09-07,2026-09-07T12:45:00.000Z,Delivered,null,0,0
5,2,Phone Case,2,598.0,2026-09-15T13:10:00.000Z,2026-09-20,2026-09-24T10:20:00.000Z,Delayed,Weather,4,1
6,3,Keyboard,1,1299.0,2026-09-03T08:55:00.000Z,2026-09-08,2026-09-08T17:30:00.000Z,Delivered,null,0,0
7,3,Monitor Stand,1,1899.0,2026-09-18T15:25:00.000Z,2026-09-23,2026-09-23T11:10:00.000Z,Delivered,null,0,0
8,3,Power Bank,1,1599.0,2026-10-01T19:05:00.000Z,2026-10-06,null,Delayed,Address Not Found,4,1
9,4,Headphones,1,2999.0,2026-09-05T12:35:00.000Z,2026-09-10,2026-09-09T15:50:00.000Z,Delivered,null,0,0
10,4,Smart Watch,1,4999.0,2026-09-20T10:00:00.000Z,2026-09-25,2026-09-30T13:15:00.000Z,Delayed,Courier Delay,5,1


In [0]:
# 3. Pipeline - calculate the latest delivery status
# ---- PIPELINE: apply the latest delivery updates to the orders ----
report_date = F.to_date(F.lit(REPORT_DATE))
 
latest = (orders.join(updates, on="order_id", how="left")
    .withColumn("delivery_date", F.coalesce(F.col("new_delivery_date"), F.col("delivery_date")))
    .withColumn("issue",         F.coalesce(F.col("new_issue"), F.col("issue")))
    .withColumn("delay_days", F.greatest(
        F.datediff(F.coalesce(F.to_date("delivery_date"), report_date), F.col("expected_date")), F.lit(0)))
    .withColumn("delayed", F.when(F.col("delay_days") > 0, 1).otherwise(0))
    .withColumn("status",
        F.when(F.col("delivery_date").isNotNull(),                         
                F.when(F.col("delay_days") > 0, "Delayed").otherwise("Delivered"))
         .when(F.col("new_status").isNotNull(), F.col("new_status"))       
         .otherwise(F.col("status")))                                      
    .select(orders.columns))                                             
 
changed = latest.join(updates.select("order_id"), on="order_id", how="inner")   
print("Orders in latest table:", latest.count(), "| Updated orders:", changed.count())
display(changed.select("order_id", "customer_id", "status", "delivery_date", "delay_days", "issue").orderBy("order_id"))

Orders in latest table: 22 | Updated orders: 5


order_id,customer_id,status,delivery_date,delay_days,issue
3,1,Delayed,2026-10-09T11:20:00.000Z,9,Courier delay
8,3,Delayed,null,4,Address not found
14,6,Delayed,2026-10-08T15:40:00.000Z,5,Stock shortage
19,2,Delivered,2026-10-09T17:10:00.000Z,0,null
22,4,Delivered,2026-10-10T10:00:00.000Z,0,null


In [0]:
# Make the updated rows available to SQL
changed.createOrReplaceTempView("order_updates")

4. Update the Delta table with MERGE (updates existing orders in place)

In [0]:
%sql
MERGE INTO customer_orders_clean AS t
USING order_updates AS u
ON t.order_id = u.order_id
WHEN MATCHED THEN UPDATE SET *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


5. Check what changed (Delta time travel: version 0 vs now)

In [0]:
%sql
DESCRIBE HISTORY customer_orders_clean

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-07T05:17:00.000Z,72240446752486,tsaravanapriya535@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(954177145644693),1dabcadb-d548-42f9-a7fd-3b5c739c2e9e,1107-044448-2d3v14mo-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 7881, p25FileSize -> 4303, numDeletionVectorsRemoved -> 1, minFileSize -> 4303, numAddedFiles -> 1, maxFileSize -> 4303, p75FileSize -> 4303, p50FileSize -> 4303, numAddedBytes -> 4303)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-07T05:16:58.000Z,72240446752486,tsaravanapriya535@gmail.com,MERGE,"Map(predicate -> [""(order_id#13871 = order_id#12819)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(954177145644693),1dabcadb-d548-42f9-a7fd-3b5c739c2e9e,1107-044448-2d3v14mo-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3627, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 5346, materializeSourceTimeMs -> 922, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2093, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2217)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-07T05:13:30.000Z,72240446752486,tsaravanapriya535@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(954177145644693),ba0620ac-cb0f-49d0-bd9a-ec76546cb7a0,1107-044448-2d3v14mo-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 22, numOutputBytes -> 4254)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT b.order_id,
b.status     AS old_status, a.status     AS new_status,
b.delay_days AS old_delay,  a.delay_days AS new_delay
FROM customer_orders_clean VERSION AS OF 0 AS b
JOIN customer_orders_clean AS a ON a.order_id = b.order_id
WHERE b.status <> a.status OR b.delay_days <> a.delay_days
ORDER BY b.order_id

order_id,old_status,new_status,old_delay,new_delay
3,Delayed,Delayed,10,9
14,Delayed,Delayed,7,5
19,In Transit,Delivered,0,0
22,In Transit,Delivered,0,0


In [0]:
# 6. Save the result as CSV too (optional)
OUT = f"{BASE}/output"
(spark.table("customer_orders_clean").orderBy("order_id")
      .coalesce(1).write.mode("overwrite").option("header", True)
      .csv(f"{OUT}/latest_delivery_status"))
print("CSV saved to", OUT)

CSV saved to /Volumes/workspace/default/customer_orders/output


7. Analysis (optional) - top 5 delayed customers

In [0]:
%sql
SELECT c.customer_name, c.region,
SUM(o.delayed)    AS delayed_orders,
SUM(o.delay_days) AS total_delay_days
FROM customer_orders_clean o
JOIN customer_customers c ON c.customer_id = o.customer_id
GROUP BY c.customer_name, c.region
HAVING SUM(o.delayed) > 0
ORDER BY delayed_orders DESC, total_delay_days DESC, c.customer_name
LIMIT 5

customer_name,region,delayed_orders,total_delay_days
Arun Kumar,South,2,12
Rahul Das,East,2,10
Meena Iyer,South,1,5
Sneha Patel,West,1,5
Karthik Raj,South,1,4


In [0]:
%sql
-- Orders by latest status
SELECT status, COUNT(*) AS orders
FROM customer_orders_clean
GROUP BY status
ORDER BY orders DESC

status,orders
Delivered,12
Delayed,9
In Transit,1
